# Demo 03 — Why retrieval is hard: a question ≠ its answer

The thought-provoking one. A human reading *"can I return an opened item?"* instantly connects it to
a policy clause that says *"unsealed merchandise may be exchanged within 14 days"* — **zero shared
words**. For a machine this is a research field. We'll watch the problem appear and then fix it.

> Needs an embeddings endpoint (`EMBED_MODEL` in `.env`; e.g. Ollama `nomic-embed-text`). The chat
> model is reused for HyDE. This notebook is about *seeing the numbers*, not the framework.

In [1]:
import os
import numpy as np

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                        # reads ../.env  (no MLflow here — this demo is pure embeddings)

client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:11434/v1"),
                api_key=os.environ.get("OPENAI_API_KEY", "not-needed-for-local"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "nomic-embed-text")
CHAT_MODEL  = os.environ.get("MODEL", "qwen2.5:7b")

def embed(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("embeddings client ready:", EMBED_MODEL)


embeddings client ready: nomic-embed-text:latest


## The setup: one question, some other questions, and the real answer

Lexical (keyword) search fails immediately — the answer shares almost no words with the question.
Let's see what *semantic* embeddings do.

In [2]:
question = "Can I return an opened item?"

other_questions = [
    "How do I track my package?",
    "What is your refund window for sealed products?",
    "Can I exchange something I already unwrapped?",   # a paraphrase-ish question
]
the_answer = ("Unsealed or opened merchandise may be exchanged within 14 days of delivery; "
              "no cash refund is given for opened items.")

q = embed([question])[0]
print("Q vs OTHER QUESTIONS (symmetric: similar shape):")
for t, e in zip(other_questions, embed(other_questions)):
    print(f"  {cos(q, e):.3f}  {t}")
print(f"\nQ vs THE ACTUAL ANSWER:")
print(f"  {cos(q, embed([the_answer])[0]):.3f}  {the_answer[:60]}...")

Q vs OTHER QUESTIONS (symmetric: similar shape):
  0.570  How do I track my package?
  0.667  What is your refund window for sealed products?
  0.687  Can I exchange something I already unwrapped?

Q vs THE ACTUAL ANSWER:
  0.743  Unsealed or opened merchandise may be exchanged within 14 da...


**The asymmetry.** Notice the question often sits *closer to other questions* than to the document
that actually answers it. Questions are short and interrogative; answers are long and declarative —
they live in **different regions of embedding space**. A general-purpose ("symmetric") model finds
look-alike *questions*, not *answers*. This is why naive "embed the query, grab nearest" disappoints.

Two production fixes:
1. **Asymmetric embedding models** trained on query→passage pairs (MS MARCO; E5 with `query:` /
   `passage:` prefixes; DPR dual encoders).
2. **HyDE** — turn the question into a *hypothetical answer*, then search with **that**. Now you're
   matching answer-to-answer.

## HyDE: search with a hypothetical answer

Ask the LLM to *write* the answer it expects, then embed that. A hypothetical answer shares the
vocabulary, length, and shape of *real* answers — so its nearest neighbours are real answers.

In [3]:
hyde_prompt = f"Write a one-sentence store-policy statement that would answer: {question}"
hypo = client.chat.completions.create(
    model=CHAT_MODEL, messages=[{"role": "user", "content": hyde_prompt}]
).choices[0].message.content.strip()
print("Hypothetical answer:\n ", hypo, "\n")

h = embed([hypo])[0]
a = embed([the_answer])[0]
print(f"Question  -> answer similarity : {cos(q, a):.3f}")
print(f"HyDE doc  -> answer similarity : {cos(h, a):.3f}   (usually higher — the gap closed)")

Hypothetical answer:
  Opened items are generally not eligible for return unless they are defective or do not match the product description; please contact customer service for specific guidance. 

Question  -> answer similarity : 0.743
HyDE doc  -> answer similarity : 0.673   (usually higher — the gap closed)


## The takeaway

Keyword fails → embeddings fix vocabulary → but Q and A live apart → asymmetric models / HyDE bridge
the shape gap. And there's more in production: **hybrid search** (add BM25 so exact strings like
`SKU AZ-4471` aren't lost) and **reranking** (a cross-encoder re-scores the shortlist). The standard
pipeline is `hybrid → RRF fusion → cross-encoder rerank → top-k → LLM`.

The point of the whole session in one line: **all this machinery just to reproduce what your brain
does in half a second when you "go look it up."** Easy for a human; a research field for a machine.

*(RAG vs fine-tuning vs long-context — when to reach for which — is Session 8.)*